In [28]:
!pip install osmnx geopandas rtree polars

In [29]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
import polars as pl
import numpy as np
import warnings

warnings.filterwarnings('ignore')
print("Libraries loaded.")

Libraries loaded.


In [30]:
# 1. Map Download
places = [
    "Western Province, Sri Lanka", "Central Province, Sri Lanka",
    "North Western Province, Sri Lanka", "Southern Province, Sri Lanka"
]
tags = {"amenity": ["school", "hospital", "bus_station", "bank"], "public_transport": ["station"]}

print("Downloading OSM POI data...")
gdf_pois = ox.features_from_place(places, tags)
gdf_pois = gdf_pois[gdf_pois.geometry.type == 'Point'][['amenity', 'geometry']].reset_index(drop=True)

# 2. Prep Outlet Data
df_master = pd.read_csv('outlet_master.csv')
df_coords = pd.read_csv('outlet_coordinates.csv')
df_outlets = pd.merge(df_master, df_coords, on='Outlet_ID', how='inner')

geometry = [Point(xy) for xy in zip(df_outlets['Longitude'], df_outlets['Latitude'])]
gdf_outlets = gpd.GeoDataFrame(df_outlets, geometry=geometry, crs="EPSG:4326")

# New step: Spatially join outlets with province boundaries to get Province information
print("Assigning Provinces to Outlets...")
# Get province boundaries
gdf_provinces = ox.geocode_to_gdf(places)
# Select only relevant columns and rename for clarity
gdf_provinces = gdf_provinces[['name', 'geometry']].rename(columns={'name': 'Province'})
# Perform a spatial join
gdf_outlets_with_province = gpd.sjoin(gdf_outlets, gdf_provinces, how="left", predicate="within")
# Keep only the Outlet_ID and Province, then merge back to df_outlets
df_outlets = pd.merge(df_outlets, gdf_outlets_with_province[['Outlet_ID', 'Province']], on='Outlet_ID', how='left')
# Fill any outlets that didn't fall into a province with 'Unknown' for robustness
df_outlets['Province'] = df_outlets['Province'].fillna('Unknown')

# 3. Spatial Math (1km Radius)
gdf_pois_proj = gdf_pois.to_crs("EPSG:32644")
gdf_outlets_proj = gdf_outlets.to_crs("EPSG:32644")
gdf_outlets_proj['geometry'] = gdf_outlets_proj.geometry.buffer(1000)

joined = gpd.sjoin(gdf_pois_proj, gdf_outlets_proj, how="inner", predicate="within")

# 4. Export Gold Layer
poi_counts = joined.groupby(['Outlet_ID', 'amenity']).size().unstack(fill_value=0).reset_index()
df_gold_outlets = pd.merge(df_outlets, poi_counts, on='Outlet_ID', how='left').fillna(0)
df_gold_outlets.to_csv('gold_outlets_with_pois.csv', index=False)
print("Spatial engineering complete.")

Assigning Provinces to Outlets...
Spatial engineering complete.


In [31]:
import polars as pl

print("Starting Polars Data Cleaning & Seasonality Smoothing...")
df_transactions = pl.scan_csv('transactions_history_final.csv')

# 1. Trap the System Ghosts
cleaned_transactions = (
    df_transactions
    .unique()
    .drop_nulls(subset=['Outlet_ID'])
    .filter(pl.col('Volume_Liters') != 0)
)

# 2. Calculate volume PER MONTH for each outlet
monthly_sales = (
    cleaned_transactions
    .group_by(["Outlet_ID", "Year", "Month", "Distributor_ID"])
    .agg(pl.col("Volume_Liters").sum().alias("Monthly_Volume"))
)

# 3. Redistribute the holiday spikes by taking the Average Monthly Volume
historical_sales = (
    monthly_sales
    .group_by("Outlet_ID")
    .agg([
        # Taking the mean redistributes the Awurudu/Christmas spikes!
        pl.col("Monthly_Volume").mean().alias("Avg_Monthly_Volume"),
        pl.col("Distributor_ID").first().alias("Distributor_ID")
    ])
    .collect()
)

historical_sales.write_csv('silver_cleaned_transactions.csv')
print("Silver layer complete. Seasonal holiday spikes have been smoothed!")

Starting Polars Data Cleaning & Seasonality Smoothing...
Silver layer complete. Seasonal holiday spikes have been smoothed!


In [32]:
import pandas as pd
import numpy as np

print("Loading Silver and Gold layers...")
sales_df = pd.read_csv('silver_cleaned_transactions.csv')
spatial_df = pd.read_csv('gold_outlets_with_pois.csv')

final_df = pd.merge(spatial_df, sales_df, on='Outlet_ID', how='left').fillna(0)

# --- MAP THE GEOGRAPHY ---
prov_map = {
    'DIST_W_01': 'Western', 'DIST_W_02': 'Western', 'DIST_W_03': 'Western',
    'DIST_C_01': 'Central', 'DIST_C_02': 'Central', 'DIST_C_03': 'Central',
    'DIST_NW_01': 'North-Western', 'DIST_NW_02': 'North-Western',
    'DIST_S_01': 'Southern', 'DIST_S_02': 'Southern'
}
final_df['Province'] = final_df['Distributor_ID'].astype(str).map(prov_map).fillna('Unknown')

# --- CLUSTER & UNCAP POTENTIAL ---
print("Calculating environment scores and clustering...")
final_df['Catchment_Score'] = final_df['school'] + final_df['bus_station'] + final_df['hospital'] + final_df['bank']
final_df['Traffic_Tier'] = pd.qcut(final_df['Catchment_Score'].rank(method='first'), q=10, labels=False)
final_df['Granular_Cluster'] = final_df['Province'] + "_" + final_df['Traffic_Tier'].astype(str)

print("Finding the Star performers using the smoothed seasonal baseline...")
# We use the smoothed Avg_Monthly_Volume to find the true Star
cluster_stars = final_df.groupby('Granular_Cluster')['Avg_Monthly_Volume'].transform(lambda x: x.quantile(0.90))

print("Uncapping latent potential for January 2026...")
final_df['Maximum_Monthly_Liters'] = np.maximum(final_df['Avg_Monthly_Volume'], cluster_stars)

# --- EXPORT ---
submission = final_df[['Outlet_ID', 'Maximum_Monthly_Liters']]
submission.to_csv('VisionAI_predictions.csv', index=False)

print("\n!!! PIPELINE FINISHED !!!")
print("Download 'VisionAI_predictions.csv' from the Colab file explorer.")
display(submission.head())

Loading Silver and Gold layers...
Calculating environment scores and clustering...
Finding the Star performers using the smoothed seasonal baseline...
Uncapping latent potential for January 2026...

!!! PIPELINE FINISHED !!!
Download 'VisionAI_predictions.csv' from the Colab file explorer.


,Outlet_ID,Maximum_Monthly_Liters
0,OUT_00001,135.503521
1,OUT_00002,139.045685
2,OUT_00003,140.911656
3,OUT_00004,131.179337
4,OUT_00005,129.587564
